# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/damlablgc/flyrankinternship/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

My lane is Refresh / Content Opportunity Scoring. `is_declining` is a binary, directly
observed outcome (built the same way as the Week-4 baseline: `imp_after < 0.8 * imp_trailing`),
not a proxy score — so per the method menu:

- "yes/no with an observed label" -> Logistic Regression first, then Random Forest.
- "which first? ranking" -> use the classifier's probability, evaluated at precision@K.

I train both: Logistic Regression (readable, linear, my baseline learner) and Random Forest
(non-linear, the challenger) on the same features the baseline rule was built on, then rank the
test set by predicted P(is_declining=1) and compute precision@50 -- the same metric the Week-4
rule was judged on. Random Forest only earns its extra complexity if the comparison table below
shows it beating Logistic Regression and the rule baseline.

In [1]:
# Label shape check: is_declining is an observed 0/1 outcome, not a proxy -- so this
# is a "yes/no with an observed label" problem per the method menu, which points at
# Logistic Regression first, then Random Forest as the stronger challenger.
# Both models' predict_proba(...)[:, 1] doubles as the ranking score precision@50 needs.
METHOD_CHOICE = "Logistic Regression (readable baseline learner) + Random Forest (challenger)"
print(f"Method choice: {METHOD_CHOICE}")
print("Ranking score for precision@K: predict_proba(X)[:, 1] -> P(is_declining=1)")

Method choice: Logistic Regression (readable baseline learner) + Random Forest (challenger)
Ranking score for precision@K: predict_proba(X)[:, 1] -> P(is_declining=1)


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Grouped split by `client_hash_id`, 80/20, `random_state=42`. Content items from the same
client share hidden client-level character (traffic scale, industry, content strategy) --
a random row split would let the model see near-duplicate clients in both train and test and
look better than it really is. A client-grouped split asks the honest deployment question:
"does this work for a client the model has never seen?" I stay on the same single month
(2026-03, DECISION_DAY 2026-03-15) and same feature/label logic as the Week-4 baseline, so the
comparison in Section 3 is apples-to-apples.

In [2]:
from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MONTH_PATH = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"
DECISION_DAY = '2026-03-15'

# Same feature + label query as the Week-4 baseline (work/notebooks/w04_baseline_score.ipynb)
# -- same month, same decision day, so Section 3's comparison is apples-to-apples.
features = con.sql(f"""
    WITH fx AS (
        SELECT
            f.client_hash_id, f.content_hash_id,
            SUM(CASE WHEN f.report_date <= DATE '{DECISION_DAY}' THEN f.gsc_impressions ELSE 0 END) AS imp_trailing,
            SUM(CASE WHEN f.report_date <= DATE '{DECISION_DAY}' THEN f.gsc_clicks ELSE 0 END)      AS clk_trailing,
            AVG(CASE WHEN f.report_date <= DATE '{DECISION_DAY}' THEN f.gsc_avg_position END)       AS pos_trailing,
            MAX(CASE WHEN f.ga4_data_available IS TRUE THEN 1 ELSE 0 END)                            AS has_ga4_data
        FROM read_parquet('{MONTH_PATH}') f
        GROUP BY 1, 2
    )
    SELECT
        fx.*,
        DATE_DIFF('day', dc.content_created_date, DATE '{DECISION_DAY}') AS content_age_days
    FROM fx
    JOIN read_parquet('{REL}/dim_content.parquet') dc
        ON fx.content_hash_id = dc.content_hash_id
""").df()

labels = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(CASE WHEN report_date > DATE '{DECISION_DAY}' THEN gsc_impressions ELSE 0 END) AS imp_after
    FROM read_parquet('{MONTH_PATH}')
    GROUP BY 1, 2
""").df()

merged = features.merge(labels, on=['client_hash_id', 'content_hash_id'])
merged['is_declining'] = (merged['imp_after'] < 0.8 * merged['imp_trailing']).astype(int)
merged['ctr_trailing'] = merged['clk_trailing'] / merged['imp_trailing'].replace(0, pd.NA)

# Same scored population as the baseline: rows with at least some trailing impressions.
scored = merged[merged['imp_trailing'] > 0].copy()
scored['ctr_trailing'] = scored['ctr_trailing'].astype(float)

FEATURE_COLS = ['imp_trailing', 'clk_trailing', 'pos_trailing', 'ctr_trailing', 'has_ga4_data', 'content_age_days']

print(f"Scored rows: {len(scored)}")
print(f"Distinct clients: {scored['client_hash_id'].nunique()}")
print(f"Base rate (is_declining=1): {scored['is_declining'].mean():.3f}")
print("\nMissing values per feature:")
print(scored[FEATURE_COLS].isna().sum())

# --- Grouped split by client_hash_id -------------------------------------
GROUP_COL = 'client_hash_id'
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(scored, groups=scored[GROUP_COL]))

train_df = scored.iloc[train_idx].copy()
test_df = scored.iloc[test_idx].copy()

train_clients = set(train_df[GROUP_COL])
test_clients = set(test_df[GROUP_COL])
overlap = train_clients & test_clients

print(f"\nTrain rows: {len(train_df)}  ({train_df[GROUP_COL].nunique()} clients)")
print(f"Test rows:  {len(test_df)}  ({test_df[GROUP_COL].nunique()} clients)")
print(f"Client overlap between train and test: {len(overlap)} (must be 0)")
assert len(overlap) == 0, "Client leakage between train and test!"

print(f"\nTrain base rate: {train_df['is_declining'].mean():.3f}")
print(f"Test base rate:  {test_df['is_declining'].mean():.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Scored rows: 151981
Distinct clients: 44
Base rate (is_declining=1): 0.327

Missing values per feature:
imp_trailing        0
clk_trailing        0
pos_trailing        0
ctr_trailing        0
has_ga4_data        0
content_age_days    0
dtype: int64

Train rows: 138771  (35 clients)
Test rows:  13210  (9 clients)
Client overlap between train and test: 0 (must be 0)

Train base rate: 0.322
Test base rate:  0.373


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Same six features (five core + `ctr_trailing`, the feature the rule itself is built on), same
train/test split, same precision@50 metric. Logistic Regression gets standardized features
(raw `imp_trailing` is in the hundreds of thousands vs `pos_trailing` in the 1-100 range --
unscaled, the fit would be dominated by magnitude, not signal). Random Forest uses raw values
(tree splits are scale-invariant). Both models rank the test set by `predict_proba(X)[:, 1]`.

The baseline rule is recomputed on this SAME test split -- not reused from the Week-4 full-
population number -- because the honest comparison needs identical held-out data for all three.

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import sklearn

RANDOM_STATE = 42
print(f"scikit-learn version: {sklearn.__version__}  (random_state={RANDOM_STATE} everywhere)")

X_train = train_df[FEATURE_COLS].values
y_train = train_df['is_declining'].values
X_test = test_df[FEATURE_COLS].values
y_test = test_df['is_declining'].values

# Logistic Regression needs scaled features -- imp_trailing runs into the hundreds
# of thousands while pos_trailing is 1-100; unscaled, the fit would be dominated by
# magnitude, not by which feature actually carries signal.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
log_reg.fit(X_train_scaled, y_train)

rf = RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=20,
    random_state=RANDOM_STATE, n_jobs=-1,
)
rf.fit(X_train, y_train)

test_df = test_df.copy()
test_df['score_logreg'] = log_reg.predict_proba(X_test_scaled)[:, 1]
test_df['score_rf'] = rf.predict_proba(X_test)[:, 1]


def precision_at_k(labels_sorted_desc, k):
    return np.asarray(labels_sorted_desc)[:k].mean()


K = 50

ranked_logreg = test_df.sort_values('score_logreg', ascending=False)
ranked_rf = test_df.sort_values('score_rf', ascending=False)

p_at_k_logreg = precision_at_k(ranked_logreg['is_declining'].values, K)
p_at_k_rf = precision_at_k(ranked_rf['is_declining'].values, K)

# --- Baseline rule, recomputed on THIS test split only (fair, same-split comparison) ---
visible_test = test_df[(test_df['imp_trailing'] >= 500) & (test_df['pos_trailing'] > 0)]
ctr_median_test = visible_test['ctr_trailing'].median()


def make_flag(row):
    is_visible = row['imp_trailing'] >= 500
    good_position = (row['pos_trailing'] > 0) and (row['pos_trailing'] <= 20)
    low_ctr = pd.notna(row['ctr_trailing']) and (row['ctr_trailing'] < ctr_median_test)
    return is_visible and good_position and low_ctr


test_df['flag_baseline'] = test_df.apply(make_flag, axis=1)
test_df['score_baseline'] = test_df['flag_baseline'].astype(int) * test_df['imp_trailing']
ranked_baseline = test_df.sort_values('score_baseline', ascending=False)
p_at_k_baseline = precision_at_k(ranked_baseline['is_declining'].values, K)

test_base_rate = test_df['is_declining'].mean()

comparison = pd.DataFrame([
    {
        'model': 'Baseline rule (Week 4)',
        f'precision_at_{K}': round(p_at_k_baseline, 3),
        'base_rate': round(test_base_rate, 3),
        'n_test_rows': len(test_df),
        'n_flagged': int(test_df['flag_baseline'].sum()),
    },
    {
        'model': 'Logistic Regression',
        f'precision_at_{K}': round(p_at_k_logreg, 3),
        'base_rate': round(test_base_rate, 3),
        'n_test_rows': len(test_df),
        'n_flagged': len(test_df),
    },
    {
        'model': 'Random Forest',
        f'precision_at_{K}': round(p_at_k_rf, 3),
        'base_rate': round(test_base_rate, 3),
        'n_test_rows': len(test_df),
        'n_flagged': len(test_df),
    },
])

print(f"\nCTR median threshold (test-split visible cohort, n={len(visible_test)}): {ctr_median_test:.4f}")
print(f"\n=== Comparison table (test split, K={K}) ===")
print(comparison.to_string(index=False))

scikit-learn version: 1.6.1  (random_state=42 everywhere)

CTR median threshold (test-split visible cohort, n=2731): 0.0034

=== Comparison table (test split, K=50) ===
                 model  precision_at_50  base_rate  n_test_rows  n_flagged
Baseline rule (Week 4)             0.20      0.373        13210       1132
   Logistic Regression             0.50      0.373        13210      13210
         Random Forest             0.54      0.373        13210      13210


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [4]:
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score

# --- Feature importances (Random Forest, built-in Gini-based) ---
rf_importances = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
print("=== Random Forest feature importances (built-in, Gini-based) ===")
print(rf_importances.round(3))

# --- Logistic Regression coefficients (on standardized features -> comparable magnitudes) ---
logreg_coefs = pd.Series(log_reg.coef_[0], index=FEATURE_COLS).sort_values(key=abs, ascending=False)
print("\n=== Logistic Regression coefficients (standardized features) ===")
print(logreg_coefs.round(3))

# --- Permutation importance on the TEST set: shuffle each feature and see how much
#     the model's ranking quality (ROC-AUC) degrades. This catches importance that's
#     just an artifact of the training fit, not real out-of-sample predictive signal.
perm = permutation_importance(
    rf, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE, scoring='roc_auc', n_jobs=-1
)
perm_importances = pd.Series(perm.importances_mean, index=FEATURE_COLS).sort_values(ascending=False)
print("\n=== Random Forest permutation importance (test set, scoring=roc_auc) ===")
print(perm_importances.round(4))

# --- Sanity check: is the top feature suspiciously perfect? (AUC near 1.0 -> likely leakage) ---
auc_rf = roc_auc_score(y_test, test_df['score_rf'])
auc_logreg = roc_auc_score(y_test, test_df['score_logreg'])
print(f"\nRF test ROC-AUC: {auc_rf:.3f}  |  Logistic Regression test ROC-AUC: {auc_logreg:.3f}")

# --- 3 concrete wrong cases: false positives in the RF top-50 ---
top50_rf = ranked_rf.head(50)
n_fp = int((top50_rf['is_declining'] == 0).sum())
false_positives = top50_rf[top50_rf['is_declining'] == 0].head(3)
cols_to_show = ['imp_trailing', 'clk_trailing', 'pos_trailing', 'ctr_trailing',
                 'has_ga4_data', 'content_age_days', 'score_rf']
print(f"\n=== False positives in RF's top 50 (flagged high risk, actually stable): {n_fp} / 50 ===")
print(false_positives[cols_to_show].round(4).to_string())

# --- For contrast: 3 concrete wrong cases the BASELINE made but RF got right ---
baseline_flagged_ids = set(test_df.loc[test_df['flag_baseline'], 'content_hash_id'])
rf_top50_ids = set(top50_rf['content_hash_id'])
baseline_only_wrong = test_df[
    test_df['content_hash_id'].isin(baseline_flagged_ids - rf_top50_ids) & (test_df['is_declining'] == 0)
].head(3)
print(f"\n=== Baseline flagged these as risky (wrong), RF's top 50 did not include them ===")
print(baseline_only_wrong[cols_to_show[:-1] + ['score_rf']].round(4).to_string())

=== Random Forest feature importances (built-in, Gini-based) ===
content_age_days    0.405
imp_trailing        0.197
pos_trailing        0.157
ctr_trailing        0.123
has_ga4_data        0.063
clk_trailing        0.055
dtype: float64

=== Logistic Regression coefficients (standardized features) ===
clk_trailing       -0.679
imp_trailing        0.278
has_ga4_data       -0.232
content_age_days   -0.162
pos_trailing       -0.105
ctr_trailing        0.045
dtype: float64

=== Random Forest permutation importance (test set, scoring=roc_auc) ===
content_age_days    0.0763
ctr_trailing        0.0249
has_ga4_data        0.0185
imp_trailing        0.0163
clk_trailing        0.0093
pos_trailing        0.0090
dtype: float64

RF test ROC-AUC: 0.636  |  Logistic Regression test ROC-AUC: 0.575

=== False positives in RF's top 50 (flagged high risk, actually stable): 23 / 50 ===
        imp_trailing  clk_trailing  pos_trailing  ctr_trailing  has_ga4_data  content_age_days  score_rf
88286        2504

**Feature importance:** `content_age_days` dominates both Random Forest's Gini importance
(0.405, more than double the runner-up) and its permutation importance on the held-out test
set (0.0763, ~3x the next feature) -- and it is measured out-of-sample, so this is real signal,
not a training-set artifact. Logistic Regression's coefficient on it is negative (-0.162):
older content is associated with a *lower* predicted decline probability.

This is the same direction Week 4's signal audit found for staleness (OPPOSITE verdict: younger
content declined more, 21.3% vs 13.9%, likely survivorship bias) -- the model has rediscovered
that association, not contradicted it. It is not leakage (ROC-AUC of 0.636 for RF and 0.575 for
LR are nowhere near the ~1.0 that flags a leaked label; the feature is known before the decision
day). But it is not a causal claim either: I read this as an observed, directional association
possibly inherited from the same survivorship effect, not "younger content causes decline" --
worth a caution note if this ships as a claim.

**Errors:** RF's top 50 has 23 false positives (27/50 correct = the reported precision@50 of
0.54). The 3 shown are young pages (33-68 days) with strong impressions and decent position but
near-zero CTR (0.03-0.12%) -- plausible navigational/branded queries where low CTR doesn't mean
disengagement, just that users already got their answer or typed the URL directly.

**Where the model earns its keep over the rule:** the 3 "baseline flagged, RF didn't" cases are
old pages (97-209 days) with good position and low CTR -- exactly what the Week-4 rule would
flag -- but they stayed stable. RF scored them low (0.27-0.32) because it also weighs age, a
distinction the position+CTR-only rule structurally cannot make.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.